# Vaccination tweet model benchmark (Kaggle GPU)
This notebook stages three open-weight model families and reproducible checkpoints. It does not upload tweets to GitHub, and does not assume automated labels are gold-standard. Use a **private** Kaggle dataset. GPU/Internet are required for model downloads and Qwen.

In [ ]:
from pathlib import Path
import os, sys, subprocess
paths = sorted(Path("/kaggle/input").rglob("*.csv"))
print("Available input CSVs (select only approved data):")
for p in paths[:30]: print(" ",p)
print("GPU runtime:", subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip())

In [ ]:
# Edit these variables after inspecting private /kaggle/input paths.
MASTER_CSV = ""  # e.g. "/kaggle/input/private-tweets/vaccination_master.csv"
ENRICHMENT_CSV = ""  # optional separate 3,208-record pool; otherwise empty
TEXT_COLUMN = "text"
DATE_COLUMN = "date"
ID_COLUMN = ""  # e.g. "tweet_id"; leave empty to use text hash
LANGUAGE_COLUMN = "language"  # verified prior language screen, or absent
MAIN_N = 2000
ENRICHMENT_N = 1000
RUN_QWEN = True
QWEN_N = 300
assert MASTER_CSV and Path(MASTER_CSV).is_file(), "Set MASTER_CSV to the private source file. No data was read."
assert not ENRICHMENT_CSV or Path(ENRICHMENT_CSV).is_file(), "Check ENRICHMENT_CSV."
print("Input configured; original data will not be changed.")

In [ ]:
repo_dir = Path("/kaggle/working/vaccination-sentiment-research")
subprocess.run(["git","clone","--depth","1","--branch","feature/kaggle-model-benchmark-v1",
 "https://github.com/keplaxo/Qualitative-analysis-of-Tweets-on-Vaccination.git",str(repo_dir)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(repo_dir)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(repo_dir/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","pytest","-q",str(repo_dir/"tests")],check=True)
print("Repository revision:",subprocess.run(["git","rev-parse","HEAD"],cwd=repo_dir,capture_output=True,text=True,check=True).stdout.strip())

In [ ]:
output = Path("/kaggle/working/vax_outputs")
output.mkdir(parents=True,exist_ok=True)
sample = output/"benchmark_sample.csv"
cmd=[sys.executable,"-m","vaxbench.cli","prepare","--input",MASTER_CSV,
     "--output",str(sample),"--text-col",TEXT_COLUMN,"--date-col",DATE_COLUMN,
     "--language-col",LANGUAGE_COLUMN,"--main-n",str(MAIN_N),
     "--enrichment-n",str(ENRICHMENT_N),"--seed","20261017"]
if ID_COLUMN: cmd+=["--id-col",ID_COLUMN]
if ENRICHMENT_CSV: cmd+=["--enrichment",ENRICHMENT_CSV]
subprocess.run(cmd,check=True)
print((Path(str(sample)+".manifest.json")).read_text())

In [ ]:
def stage(*args):
    command=[sys.executable,"-m","vaxbench.cli",*args]
    print("Running:", " ".join(command[:5]),"...")
    return subprocess.run(command, check=True)
stage("sentiment","--model","multilingual","--input",str(sample),
      "--output",str(output/"xlmt_sentiment.jsonl"))
stage("sentiment","--model","english","--input",str(sample),
      "--output",str(output/"roberta_english_sentiment.jsonl"))
stage("zeroshot","--input",str(sample),
      "--output",str(output/"deberta_stance.jsonl"))

In [ ]:
triage_file = output/"qwen_triage.csv"
if RUN_QWEN:
    subprocess.run([sys.executable,"-m","vaxbench.triage",
        "--input",str(sample),"--zeroshot",str(output/"deberta_stance.jsonl"),
        "--output",str(triage_file),"--max-records",str(QWEN_N),
        "--hard-fraction","0.6"],check=True)
    try:
        stage("qwen","--input",str(triage_file),
              "--output",str(output/"qwen_semantics.jsonl"),
              "--max-records",str(QWEN_N),"--four-bit")
    except Exception as exc:
        print("Qwen not completed; prior checkpoint files remain.", repr(exc))
        QWEN_ERROR = repr(exc)
else:
    print("Qwen deliberately skipped; earlier stages retained.")

In [ ]:
import json
for p in sorted(output.glob("*.jsonl")):
    count = sum(1 for line in p.open(encoding="utf-8") if line.strip())
    print(p.name,":",count,"records")
print("Save this Kaggle notebook version and outputs as a private artifact.")
print("No independent vaccine-stance accuracy estimate has been produced.")

In [ ]:
subprocess.run([sys.executable,"-m","vaxbench.summarize",
    "--sample",str(sample),
    "--multilingual",str(output/"xlmt_sentiment.jsonl"),
    "--english",str(output/"roberta_english_sentiment.jsonl"),
    "--zeroshot",str(output/"deberta_stance.jsonl"),
    "--qwen",str(output/"qwen_semantics.jsonl"),
    "--output",str(output/"summary.json")],check=True)